# MLflow Models: Custom PyFunc Models

In this notebook we will cover:
1. Creating custom PyFunc models
2. Models with preprocessing
3. Ensemble models
4. Models with artifacts
5. Advanced predict() logic

## What is PyFunc?

**PyFunc** is a universal interface that allows:
- Wrapping any Python code as MLflow model
- Adding custom preprocessing/postprocessing
- Creating complex ensembles
- Using external artifacts (files, configs, etc.)
- Framework-independent deployment

## 1. Import Libraries

In [ ]:
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pickle
import json
import sys
import os

sys.path.append('../src')
from data_loader import load_sample_data
from model_inspector import inspect_model_structure

print(f"MLflow version: {mlflow.__version__}")

## 2. Data Preparation

In [ ]:
X_train, X_test, y_train, y_test, feature_names, target_names = load_sample_data('iris')

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Classes: {target_names}")

## 3. Simple Custom Model with Preprocessing

In [ ]:
class ModelWithPreprocessing(mlflow.pyfunc.PythonModel):
    """
    Custom model with integrated preprocessing
    """
    
    def __init__(self, model, scaler):
        self.model = model
        self.scaler = scaler
    
    def predict(self, context, model_input):
        print(f"Input shape: {model_input.shape}")
        
        scaled_input = self.scaler.transform(model_input)
        print(f"After scaling: mean={scaled_input.mean():.4f}, std={scaled_input.std():.4f}")
        
        predictions = self.model.predict(scaled_input)
        
        return predictions

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

base_model = RandomForestClassifier(n_estimators=100, random_state=42)
base_model.fit(X_train_scaled, y_train)

custom_model = ModelWithPreprocessing(model=base_model, scaler=scaler)

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("04_Custom_PyFunc_Models")

with mlflow.start_run(run_name="model_with_preprocessing") as run:
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = custom_model.predict(None, X_sample)
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=custom_model,
        signature=signature
    )
    
    run_id_preprocessing = run.info.run_id
    
    X_test_df = pd.DataFrame(X_test, columns=feature_names)
    y_pred_test = custom_model.predict(None, X_test_df)
    accuracy = (y_pred_test == y_test).mean()
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"\nRun ID: {run_id_preprocessing}")
    print(f"Accuracy: {accuracy:.4f}")

## 4. Weighted Ensemble Model

In [ ]:
class WeightedEnsemble(mlflow.pyfunc.PythonModel):
    """
    Ensemble with weighted voting
    """
    
    def __init__(self, models, weights):
        self.models = models
        self.weights = np.array(weights)
        self.weights = self.weights / self.weights.sum()  # normalize
    
    def predict(self, context, model_input):
        predictions = []
        
        for i, model in enumerate(self.models):
            pred = model.predict_proba(model_input)
            weighted_pred = pred * self.weights[i]
            predictions.append(weighted_pred)
        
        ensemble_proba = np.sum(predictions, axis=0)
        final_predictions = np.argmax(ensemble_proba, axis=1)
        
        return final_predictions

model1 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model1.fit(X_train, y_train)
acc1 = model1.score(X_test, y_test)

model2 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model2.fit(X_train, y_train)
acc2 = model2.score(X_test, y_test)

model3 = LogisticRegression(max_iter=1000, random_state=42)
model3.fit(X_train, y_train)
acc3 = model3.score(X_test, y_test)

print(f"Model 1 accuracy: {acc1:.4f}")
print(f"Model 2 accuracy: {acc2:.4f}")
print(f"Model 3 accuracy: {acc3:.4f}")

ensemble = WeightedEnsemble(
    models=[model1, model2, model3],
    weights=[acc1, acc2, acc3]  # weights based on accuracy
)

print(f"\nEnsemble weights: {ensemble.weights}")

In [ ]:
with mlflow.start_run(run_name="weighted_ensemble") as run:
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = ensemble.predict(None, X_sample)
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ensemble,
        signature=signature
    )
    
    run_id_ensemble = run.info.run_id
    
    X_test_df = pd.DataFrame(X_test, columns=feature_names)
    y_pred_test = ensemble.predict(None, X_test_df)
    accuracy = (y_pred_test == y_test).mean()
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.log_param("num_models", 3)
    mlflow.log_param("weights", ensemble.weights.tolist())
    
    print(f"\nRun ID: {run_id_ensemble}")
    print(f"Ensemble Accuracy: {accuracy:.4f}")
    print(f"Improvement over best single model: {accuracy - max(acc1, acc2, acc3):.4f}")

## 5. Model with External Artifacts (load_context)

In [ ]:
class ModelWithArtifacts(mlflow.pyfunc.PythonModel):
    """
    Model that loads configuration and preprocessing from artifacts
    """
    
    def load_context(self, context):
        """
        Called when model is loaded. Loads artifacts.
        """
        print("Loading model and artifacts...")
        
        with open(context.artifacts["model"], "rb") as f:
            self.model = pickle.load(f)
        
        with open(context.artifacts["scaler"], "rb") as f:
            self.scaler = pickle.load(f)
        
        with open(context.artifacts["config"], "r") as f:
            self.config = json.load(f)
        
        print(f"Model loaded: {type(self.model).__name__}")
        print(f"Config: {self.config}")
    
    def predict(self, context, model_input):
        use_scaling = self.config.get("use_scaling", True)
        
        if use_scaling:
            model_input = self.scaler.transform(model_input)
        
        predictions = self.model.predict(model_input)
        
        if self.config.get("return_proba", False):
            predictions = self.model.predict_proba(model_input)
        
        return predictions

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

base_model = RandomForestClassifier(n_estimators=100, random_state=42)
base_model.fit(X_train_scaled, y_train)

os.makedirs("../data/temp", exist_ok=True)
with open("../data/temp/model.pkl", "wb") as f:
    pickle.dump(base_model, f)

with open("../data/temp/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

config = {
    "use_scaling": True,
    "return_proba": False,
    "model_type": "RandomForest",
    "version": "1.0"
}
with open("../data/temp/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Artifacts created successfully")

In [ ]:
with mlflow.start_run(run_name="model_with_artifacts") as run:
    artifacts = {
        "model": "../data/temp/model.pkl",
        "scaler": "../data/temp/scaler.pkl",
        "config": "../data/temp/config.json"
    }
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    signature = infer_signature(X_sample, y_train[:5])
    
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ModelWithArtifacts(),
        artifacts=artifacts,
        signature=signature
    )
    
    run_id_artifacts = run.info.run_id
    print(f"\nRun ID: {run_id_artifacts}")

### Test model with artifacts

In [ ]:
loaded_model = mlflow.pyfunc.load_model(f"runs:/{run_id_artifacts}/model")

X_test_df = pd.DataFrame(X_test[:5], columns=feature_names)
predictions = loaded_model.predict(X_test_df)

print("Predictions:")
print(predictions)
print(f"\nActual labels: {y_test[:5]}")

### Inspect model structure

In [ ]:
print("Model with artifacts structure:")
inspect_model_structure(f"runs:/{run_id_artifacts}/model", verbose=True)

## 6. Model with Postprocessing

In [ ]:
class ModelWithPostprocessing(mlflow.pyfunc.PythonModel):
    """
    Model that returns rich output with probabilities and confidence
    """
    
    def __init__(self, model, class_names):
        self.model = model
        self.class_names = class_names
    
    def predict(self, context, model_input):
        probabilities = self.model.predict_proba(model_input)
        predictions = np.argmax(probabilities, axis=1)
        confidence = np.max(probabilities, axis=1)
        
        results = pd.DataFrame({
            'predicted_class': [self.class_names[p] for p in predictions],
            'confidence': confidence,
            'is_confident': confidence > 0.8
        })
        
        for i, class_name in enumerate(self.class_names):
            results[f'prob_{class_name}'] = probabilities[:, i]
        
        return results

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

rich_model = ModelWithPostprocessing(model=model, class_names=target_names)

In [ ]:
X_test_df = pd.DataFrame(X_test[:10], columns=feature_names)
rich_predictions = rich_model.predict(None, X_test_df)

print("Rich predictions:")
print(rich_predictions)

## 7. Comparing Custom Models

In [ ]:
from model_inspector import compare_models

print("Comparing preprocessing model vs ensemble:")
compare_models(
    uri1=f"runs:/{run_id_preprocessing}/model",
    uri2=f"runs:/{run_id_ensemble}/model"
)

## Conclusions

In this notebook we learned:

1.  Creating custom PyFunc models
2.  Integrating preprocessing in model
3.  Building weighted ensembles
4.  Using artifacts with `load_context()`
5.  Adding custom postprocessing logic
6.  Returning complex outputs (DataFrames)

### Key Patterns:
- `__init__()` - initialize model with dependencies
- `load_context()` - load artifacts when model is loaded
- `predict()` - custom prediction logic
- Artifacts - external files (configs, preprocessors, etc.)

### Use Cases:
- Preprocessing/postprocessing integration
- Complex ensemble strategies
- Models with external dependencies
- Business logic in predictions

### Next Steps:
- **Notebook 05**: Model Deployment and Serving